In [1]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast, GPT2Config
from transformers import get_linear_schedule_with_warmup

import torch
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import random_split, RandomSampler, SequentialSampler

import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
# model_name: ['gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl']
model_name = "gpt2-large" 
model_save_path = './model'

/u1/kfountou/.conda/envs/the_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
configuration = GPT2Config.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name, config=configuration)

tokenizer = GPT2TokenizerFast.from_pretrained(model_name)

model = model.to(device)

In [3]:
from nlp_dataset import generate_sample

def form_string(sample: tuple[list, float], isTrain: bool) ->tuple[str, float]:
    sample_text = sample[0]
    sample_ans = sample[1]
    input_list = [x if type(x) == str else format(x, '05.2f') for x in sample_text[:-1]]
    prompt = "<|startoftext|>" + ", ".join(input_list) + ". " + sample_text[-1] + "."
    if isTrain:
        prompt += " Answer: " + format(sample_ans, '05.2f')
        prompt += "<|endoftext|>"
    else:
        prompt += " Answer: "
    return prompt, sample_ans

In [4]:
num_cats = 8
query_type = "min"
num_query_cats = 4
train_low = 0
train_high = 5
test_low = 0
test_high = 20
num_train_samples = 500000
num_test_samples = 100

In [5]:
train_data_comb = [form_string(generate_sample(num_cats, query_type, train_low, train_high, num_query_cats), True) for _ in range(num_train_samples)]
train_data = [x[0] for x in train_data_comb]
train_data_ans = [x[1] for x in train_data_comb]
test_data_comb = [form_string(generate_sample(num_cats, query_type, test_low, test_high, num_query_cats, train=False), False) for _ in range(num_test_samples)]
test_data = [x[0] for x in test_data_comb]
test_data_ans = [x[1] for x in test_data_comb]

In [6]:
print(train_data[0])
print(test_data[0])

<|startoftext|>Catǹ, Cat+ȃ, 04.85, Catǻ, 04.87, Catǽ, Cat-ȁ, 04.89, CatǾ, 04.94, CatȀ, 04.86, Catȁ, 04.57, Catȃ, 04.66, CatȄ, 04.79. Find min of categories Catǹ, CatǾ, Catȁ and CatȄ. Answer: 04.57<|endoftext|>
<|startoftext|>Catȋ, 04.98, CatȎ, 04.69, Catȏ, Cat_Ȏ, 04.81, CatȐ, Cat*ȝ, 04.86, CatȒ, 04.25, Catș, 04.26, CatȜ, 05.07, Catȝ, 05.24. Find min of categories CatȐ, CatȎ, Catȝ and Catȋ. Answer: 


In [7]:
tokenizer = GPT2TokenizerFast.from_pretrained(model_name,
                                              bos_token='<|startoftext|>',
                                              eos_token='<|endoftext|>',
                                              unk_token='<|unknown|>',
                                              pad_token='<|pad|>'
                                             )

In [8]:
batch_size = 2
max_length = 101

# standard PyTorch approach of loading data in using a Dataset class.
class NAR_Dataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.input_ids = []
        self.attn_masks = []

        for recipe in data:
            encodings = tokenizer.encode_plus(recipe,
                                              truncation=True,
                                              padding='max_length',
                                              max_length=max_length,
                                              # return a PyTorch tensor
                                              return_tensors='pt'       
                                             )
            self.input_ids.append(torch.squeeze(encodings['input_ids'],0))
            self.attn_masks.append(torch.squeeze(encodings['attention_mask'],0))


    def __len__(self):
        return len(self.data)

    def __getitem__(self,idx):
        return self.input_ids[idx], self.attn_masks[idx]

dataset_indist = NAR_Dataset(train_data, tokenizer)
dataset_ood = NAR_Dataset(test_data, tokenizer)
print(f"input_ids: {dataset_ood[0][0]} attn_masks: {dataset_ood[0][1]}")

input_ids: tensor([50257, 21979,   132,   233,    11,  8702,    13,  4089,    11,  5181,
          132,   236,    11,  8702,    13,  3388,    11,  5181,   132,   237,
           11,  5181,    62,   132,   236,    11,  8702,    13,  6659,    11,
         5181,   132,   238,    11,  5181,     9,   132,   251,    11,  8702,
           13,  4521,    11,  5181,   132,   240,    11,  8702,    13,  1495,
           11,  5181,   132,   247,    11,  8702,    13,  2075,    11,  5181,
          132,   250,    11,  8870,    13,  2998,    11,  5181,   132,   251,
           11,  8870,    13,  1731,    13,  9938,   949,   286,  9376,  5181,
          132,   238,    11,  5181,   132,   236,    11,  5181,   132,   251,
          290,  5181,   132,   233,    13, 23998,    25,   220, 50259, 50259,
        50259]) attn_masks: tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1

In [9]:
print(tokenizer.decode(dataset_indist[0][0]))

<|startoftext|>Catǹ, Cat+ȃ, 04.85, Catǻ, 04.87, Catǽ, Cat-ȁ, 04.89, CatǾ, 04.94, CatȀ, 04.86, Catȁ, 04.57, Catȃ, 04.66, CatȄ, 04.79. Find min of categories Catǹ, CatǾ, Catȁ and CatȄ. Answer: 04.57<|endoftext|>


In [10]:
print(tokenizer.decode(dataset_indist[10][0]))

<|startoftext|>CatǸ, 01.70, Catǻ, 02.04, CatǼ, 01.95, Catǽ, 01.94, Catǿ, 02.09, CatȂ, 02.03, CatȆ, 02.13, Cat-ǻ, Catȇ, 02.64, Cat+Ǹ. Find min of categories Catǿ, CatǼ, CatȂ and CatǸ. Answer: 01.70<|endoftext|>


In [11]:
# Split into training and validation sets
train_size = int(0.9 * len(dataset_indist))
val_size = len(dataset_indist) - train_size

train_dataset, val_dataset = random_split(dataset_indist, [train_size, val_size])

# Create the DataLoaders for our training and validation datasets.
# Get training samples in random order.
train_dataloader = DataLoader(
            train_dataset, 
            sampler = RandomSampler(train_dataset),
            batch_size = batch_size # Trains with this batch size.
        )

# Get valiation samples sequentially.
validation_dataloader = DataLoader(
            val_dataset, 
            sampler = SequentialSampler(val_dataset),
            batch_size = batch_size # Evaluate with this batch size.
        )

test_dataloader = DataLoader(
            dataset_ood, 
            sampler = SequentialSampler(dataset_ood),
            batch_size = batch_size # Evaluate with this batch size.
        )
            


In [12]:
configuration = GPT2Config.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name, config=configuration)
model = model.to(device)
model.resize_token_embeddings(len(tokenizer))

epochs = 3
learning_rate = 2e-5
warmup_steps = 1e2
# to prevent any division by zero in the implementation
epsilon = 1e-8
optim = AdamW(model.parameters(), lr = learning_rate, eps = epsilon)

total_steps = len(train_dataloader) * epochs  # [no batches] x [no epochs]

# Create the learning rate scheduler.
scheduler = get_linear_schedule_with_warmup(optim,
                                            num_warmup_steps=warmup_steps,
                                            num_training_steps=total_steps)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [13]:
def infer(prompt):
    input = prompt
    input = tokenizer(input, return_tensors="pt")
    input_ids      = input["input_ids"]
    attention_mask = input["attention_mask"]

    output = model.generate(input_ids.to(device),
                            attention_mask=attention_mask.to(device),
                            max_new_tokens=5,
                            do_sample = True, top_k = 50, top_p = 0.85)
    output = tokenizer.decode(output[0], skip_special_tokens=True)
    start_index = output.find("Answer: ") + len("Answer: ")
    # Extract the first 5 characters from that point
    result = output[start_index:start_index + 5]

    return result

In [14]:
for epoch_i in range(0, epochs):
    total_train_loss = 0
    model.train() 

    for step, batch in enumerate(train_dataloader): 
        b_input_ids = batch[0].to(device) 
        b_labels    = batch[0].to(device)
        b_masks     = batch[1].to(device) 

        model.zero_grad()
        outputs = model( input_ids = b_input_ids, labels = b_labels,
                         attention_mask = b_masks, token_type_ids = None )

        loss = outputs[0]

        # Get sample every x batches.
        if step % 100 == 0 and not step == 0:
            model.eval()
            print(infer(test_data[0]))
            model.train()

        loss.backward()
        optim.step()
        scheduler.step()

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


ȁ.04.


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Ȉ.26


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Ȅ, Ca


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


ȇ, Ca


KeyboardInterrupt: 

In [25]:
print(infer(train_data[10]))

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


01.70


In [26]:
print(train_data_ans[10])

1.7


In [ ]:
input_string = infer(train_data[4])
start_index = input_string.find("Answer: ") + len("Answer: ")

# Extract the first 5 characters from that point
result = input_string[start_index:start_index + 5]

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


In [ ]:
print(result)

03.52


In [ ]:
test_data[0]

'<|startoftext|>CatȊ, 08.31, Catȋ, 06.62, Catȍ, Cat_Ȑ, 07.46, CatȎ, 07.88, CatȐ, 06.63, CatȖ, 08.78, CatȘ, 06.55, Cat_ț, Catț, 08.18. Find min of categories Catȋ, Catț, CatȘ and Catȍ. Answer: '